# Benchmark: LDS Discrepancia Ponderada vs Beam Search

Compara `DLTSLDSWDiscSolver` (discrepancia ponderada por log-prob) contra Beam Search W=32.

**Diferencia respecto a LDS-batch-d5:**  
En lugar de `child_disc += rank` (0, 1, 2, ...), usa `child_disc += -log(prob/max_prob)`.  
Esto prioriza rutas que se desvían *poco* del greedy en términos de probabilidad,
no simplemente en número de posición en el ranking.

**Solvers:**
- **LDS-wdisc-d5**: discrepancia ponderada, `max_disc=5`
- **LDS-rank-d5**: discrepancia por rank (versión anterior), `max_disc=5`
- **Beam-W32**: Beam Search W=32

Modelos: `v2_actions_rl` (branching) + `v2_cost` (bounding).

In [1]:
import sys, os, json, copy, time
import torch
import numpy as np

MAIN_SRC  = os.path.abspath('../src')
REPO_SRC  = os.path.abspath('../Repo_Oscar/CPMP-Framework/src')
MODEL_DIR = os.path.abspath('../models')

sys.path.insert(0, MAIN_SRC)

from training.training import load_model
from solvers.dlts_lds_batched import DLTSLDSBatchedSolver
from solvers.dlts_lds_wdisc import DLTSLDSWDiscSolver
from solvers.beam_search import BeamSearchSolver
from cpmp.layout import read_file
from settings import INSTANCE_FOLDER

print('Main src OK')

for k in [k for k in sys.modules if k == 'generation' or k.startswith('generation.')]:
    del sys.modules[k]
sys.path.insert(0, REPO_SRC)

from models.actions.cpmp_transformer_V2 import CPMPTransformer as OscarCPMPTransformer
from models.cost.cost_predictor_V2 import CostPredictorTransformer
from generation.adapters.input.enriched_layout_adapter import EnrichedLayoutAdapter
from generation.adapters.input.layout.layout_4D_adapter_V2 import Layout4DAdapterV2
from generation.adapters.input.stack_features.stack_features_adapter_V1 import StackFeaturesAdapterV1

print('Repo_Oscar OK')

Main src OK
Repo_Oscar OK


In [2]:
class OscarV2LayoutAdapter:
    def __init__(self, S_max, H_max):
        self._inner = EnrichedLayoutAdapter(
            layout_adapter=Layout4DAdapterV2,
            stack_features_adapter=StackFeaturesAdapterV1,
            S_max=S_max, H_max=H_max
        )
        self.S_max = S_max
    def layout_2_vec(self, layout, H):
        return self._inner.input_2_vec(layout, H)

class OscarV2Wrapper(torch.nn.Module):
    def __init__(self, model, S_max):
        super().__init__()
        self._model = model
        self.S_max  = S_max
        self.hyperparams = model.hyperparams

    def forward(self, L, X, S, H):
        logits_full = self._model(L, X, S, H)
        S_int = int(S.flatten()[0].item())
        batch = logits_full.shape[0]
        out   = torch.zeros(batch, S_int * (S_int - 1), device=logits_full.device)
        for src in range(S_int):
            for dst in range(S_int):
                if src == dst:
                    continue
                d_off = dst if dst < src else dst - 1
                oscar_idx  = src * (self.S_max - 1) + d_off
                solver_idx = src * (S_int - 1)      + d_off
                out[:, solver_idx] = logits_full[:, oscar_idx]
        return out

S_MAX, H_MAX = 10, 12
_oscar_base     = load_model(OscarCPMPTransformer, 'v2_actions_rl')
branching_model = OscarV2Wrapper(_oscar_base, S_max=S_MAX)
branching_model.layout_adapter = OscarV2LayoutAdapter(S_max=S_MAX, H_max=H_MAX)
branching_model.eval()

def load_cost_model(name):
    hp = json.load(open(os.path.join(MODEL_DIR, 'hyperparameters', f'{name}.json')))
    m = CostPredictorTransformer(**hp)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'{name}.pth'),
                                  weights_only=True, map_location='cpu'))
    m.eval()
    return m

bounding_model   = load_cost_model('v2_cost')
bounding_adapter = EnrichedLayoutAdapter(
    layout_adapter=Layout4DAdapterV2,
    stack_features_adapter=StackFeaturesAdapterV1,
    S_max=S_MAX, H_max=H_MAX
)

print(f'Modelos cargados.')

Modelos cargados.


In [3]:
T_LIM  = 15.0
W      = 32
SHARED = dict(branching_model=branching_model,
              bounding_model=bounding_model,
              bounding_adapter=bounding_adapter,
              p=0.3, k=3, d=0.8, z=0, time_limit=T_LIM)

solvers = {
    'LDS-wdisc-d5': DLTSLDSWDiscSolver(**SHARED,   max_disc=5),
    'LDS-rank-d5' : DLTSLDSBatchedSolver(**SHARED, max_disc=5),
    'Beam-W32'    : BeamSearchSolver(branching_model=branching_model,
                                     beam_width=W, expansions_per_state=W,
                                     time_limit=T_LIM),
}

for name, s in solvers.items():
    print(f'  {name:<15}: {s.name}')

  LDS-wdisc-d5   : DLTSLDSWDiscSolver
  LDS-rank-d5    : DLTSLDSBatchedSolver
  Beam-W32       : BeamSearchSolver


In [4]:
def avg(vals, mask=None):
    data = [v for v, ok in zip(vals, mask or [True]*len(vals)) if ok]
    return sum(data) / len(data) if data else float('nan')

def run_solver(solver, files, H_inf, max_steps):
    solved_list, steps_list, time_list = [], [], []
    for path in files:
        layout = read_file(path, H_inf)
        t0 = time.perf_counter()
        try:
            solved, steps = solver.solve_from_layout(layout, H_inf, max_steps)
        except Exception as e:
            print(f'  ERROR en {os.path.basename(path)}: {e}')
            solved, steps = False, max_steps
        elapsed = time.perf_counter() - t0
        solved_list.append(solved)
        steps_list.append(steps)
        time_list.append(elapsed)
    return solved_list, steps_list, time_list

print('OK')

OK


## Benchmark por categorías CVS

In [ ]:
CVS_PATH  = INSTANCE_FOLDER / 'benchmarks' / 'CVS'
MAX_STEPS = 100
N_PER_CAT = 20

ALL_CATS = sorted([d for d in os.listdir(CVS_PATH)
                   if (CVS_PATH / d).is_dir() and not d.startswith('10-')])

solver_names = list(solvers.keys())
SEP = '-' * 110

header = f"{'Cat':>6} {'H':>3} {'S':>3}"
for name in solver_names:
    header += f"  {name:>14} {'paso':>5} {'t(s)':>6}"
print(header)
print(SEP)

all_results = {}

for folder_name in ALL_CATS:
    H_r, S_r = [int(x) for x in folder_name.split('-')]
    H_inf = H_r + 2
    folder_path = CVS_PATH / folder_name
    files = sorted([str(folder_path / f)
                    for f in os.listdir(folder_path) if f.endswith('.dat')])[:N_PER_CAT]
    if not files:
        continue

    cat_res = {}
    for name, solver in solvers.items():
        s, st, t = run_solver(solver, files, H_inf, MAX_STEPS)
        cat_res[name] = dict(solved=s, steps=st, time=t)

    all_results[folder_name] = cat_res
    n = len(files)

    row = f"{folder_name:>6} {H_r:>3} {S_r:>3}"
    for name in solver_names:
        r = cat_res[name]
        row += f"  {sum(r['solved']):>3}/{n:<2} {avg(r['steps'], r['solved']):>8.1f} {avg(r['time'], r['solved']):>6.2f}"
    print(row)

print(SEP)

totals = {name: dict(solved=[], steps=[], time=[]) for name in solver_names}
for cat_res in all_results.values():
    for name in solver_names:
        totals[name]['solved'].extend(cat_res[name]['solved'])
        totals[name]['steps'].extend(cat_res[name]['steps'])
        totals[name]['time'].extend(cat_res[name]['time'])

n_tot = len(totals[solver_names[0]]['solved'])
row = f"{'TOTAL':>6} {'':>3} {'':>3}"
for name in solver_names:
    r = totals[name]
    row += f"  {sum(r['solved']):>3}/{n_tot:<2} {avg(r['steps'], r['solved']):>8.1f} {avg(r['time'], r['solved']):>6.2f}"
print(row)

   Cat   H   S    LDS-wdisc-d5  paso   t(s)     LDS-rank-d5  paso   t(s)        Beam-W32  paso   t(s)
--------------------------------------------------------------------------------------------------------------
   3-3   3   3   20/20     10.3   0.07   20/20     10.3   0.06   20/20      9.8   0.47
   3-4   3   4   20/20      8.8   0.07   20/20      8.8   0.05   20/20      8.8   0.43
   3-5   3   5   20/20     10.6   0.13   20/20     10.6   0.09   20/20     10.4   0.63
   3-6   3   6   20/20     11.8   0.71   20/20     11.8   0.20   20/20     11.7   0.87


## Resumen

In [ ]:
mask_all = [all(totals[n]['solved'][i] for n in solver_names) for i in range(n_tot)]
n_common = sum(mask_all)

ref_name = 'LDS-wdisc-d5'
ref_steps = [s for s, ok in zip(totals[ref_name]['steps'], mask_all) if ok]
ref_time  = avg(totals[ref_name]['time'], totals[ref_name]['solved'])

print(f'Instancias comunes: {n_common}/{n_tot}')
print()
print(f'{"Solver":<16} {"Avg pasos":>10} {"vs wdisc":>10} {"Avg t(s)":>10}')
print('-' * 50)
for name in solver_names:
    r = totals[name]
    steps_c = [s for s, ok in zip(r['steps'], mask_all) if ok]
    vs = (np.mean(steps_c) / np.mean(ref_steps) - 1) * 100
    marker = '  <- ref' if name == ref_name else ''
    print(f'{name:<16} {np.mean(steps_c):>10.2f} {vs:>+9.1f}% {avg(r["time"], r["solved"]):>10.3f}{marker}')

print()
print(f'Comparacion por pares (wdisc vs cada uno):')
wdisc_steps = [s for s, ok in zip(totals[ref_name]['steps'], mask_all) if ok]
for name in solver_names:
    if name == ref_name:
        continue
    other_steps = [s for s, ok in zip(totals[name]['steps'], mask_all) if ok]
    better = sum(1 for a, b in zip(wdisc_steps, other_steps) if a < b)
    equal  = sum(1 for a, b in zip(wdisc_steps, other_steps) if a == b)
    worse  = sum(1 for a, b in zip(wdisc_steps, other_steps) if a > b)
    print(f'  wdisc vs {name:<12}: mejor {better:>4} ({better/n_common:>5.1%})  '
          f'igual {equal:>4} ({equal/n_common:>5.1%})  '
          f'peor {worse:>4} ({worse/n_common:>5.1%})')

## Visualización

In [ ]:
import matplotlib.pyplot as plt

cats_sorted = sorted(all_results.keys())
colors = {'LDS-wdisc-d5': '#e05c00', 'LDS-rank-d5': '#70ad47', 'Beam-W32': '#5b9bd5'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 1. Pasos promedio por categoría ──────────────────────────────────────────
ax = axes[0]
x = np.arange(len(cats_sorted))
n = len(solver_names)
width = 0.7 / n
offsets = np.linspace(-(n-1)/2, (n-1)/2, n) * width

for i, name in enumerate(solver_names):
    vals = [avg(all_results[c][name]['steps'], all_results[c][name]['solved'])
            for c in cats_sorted]
    ax.bar(x + offsets[i], vals, width, label=name, color=colors[name], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(cats_sorted, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Avg pasos')
ax.set_title('Pasos promedio por categoria')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# ── 2. Scatter: wdisc vs rank (pasos por instancia) ───────────────────────────
ax2 = axes[1]
wdisc_s = [s for s, ok in zip(totals['LDS-wdisc-d5']['steps'], mask_all) if ok]
rank_s  = [s for s, ok in zip(totals['LDS-rank-d5']['steps'],  mask_all) if ok]
beam_s  = [s for s, ok in zip(totals['Beam-W32']['steps'],     mask_all) if ok]

ax2.scatter(rank_s,  wdisc_s, alpha=0.4, s=15, color='#70ad47', label='vs LDS-rank-d5')
ax2.scatter(beam_s,  wdisc_s, alpha=0.4, s=15, color='#5b9bd5', label='vs Beam-W32')
mx = max(max(rank_s + beam_s), max(wdisc_s)) * 1.05
ax2.plot([0, mx], [0, mx], 'r--', lw=1.5, label='igual')
ax2.set_xlabel('Pasos comparado')
ax2.set_ylabel('Pasos LDS-wdisc-d5')
ax2.set_title('wdisc vs rank y vs Beam\n(bajo diagonal = wdisc mejor)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()